In [1]:
import os
from pathlib import Path

import boto3
from botocore.exceptions import BotoCoreError, ClientError


def load_env_file():
    """Puts the variables of the project's .env into os.environ: the first .env found from this folder upwards. Nothing is printed."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if env_file.is_file():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip().removeprefix("export ").strip()] = value.strip().strip("\"'")
            return env_file
    raise FileNotFoundError("no .env found from this folder upwards")


def first_env(names, default=None):
    """The value of the first of these variables that is set. Without a default, none set is an error."""
    for name in names:
        if os.getenv(name):
            return os.getenv(name)
    if default is None:
        raise KeyError(f"none of {', '.join(names)} found in .env")
    return default


def make_aws_session():
    """boto3 session with the AWS credentials from .env. The token is only needed for temporary keys (SSO, assumed roles). The region is AWS_DEFAULT_REGION or AWS_REGION, and us-east-1 when neither is set."""
    return boto3.Session(
        aws_access_key_id=first_env(["AWS_ACCESS_KEY_ID", "AWS_ACCESS_KEY"]),
        aws_secret_access_key=first_env(["AWS_SECRET_ACCESS_KEY", "AWS_SECRET_KEY"]),
        aws_session_token=first_env(["AWS_SESSION_TOKEN"], default="") or None,
        region_name=first_env(["AWS_DEFAULT_REGION", "AWS_REGION"], default="us-east-1"),
    )


def test_aws(session):
    """Asks AWS who these credentials belong to: one call that needs no permission, so it only fails when the keys or the region are wrong."""
    try:
        identity = session.client("sts").get_caller_identity()
        print(f"OK [{session.region_name}]: AWS credentials accepted")
        print(f"account: {identity['Account']} | identity: {identity['Arn']}")
    except ClientError as e:
        error = e.response["Error"]
        print(f"ERROR {error['Code']}: {error['Message']}")
    except BotoCoreError as e:
        print(f"Connection ERROR: {e}")


load_env_file()
aws_session = make_aws_session()
test_aws(aws_session)


OK [us-east-1]: AWS credentials accepted
account: 123456789012 | identity: arn:aws:iam::123456789012:user/undercover


In [18]:
from dotenv import load_dotenv, set_key
load_dotenv(override=True)

session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

Working as arn:aws:iam::123456789012:root in us-east-1


In [19]:
agentcore = session.client("bedrock-agentcore-control")
agentcore_runtime = session.client("bedrock-agentcore")
cloudformation = session.client("cloudformation")
logs = session.client("logs")

In [2]:
# The model to try. Change it for any id the list below shows for your region.
BEDROCK_MODEL = "anthropic.claude-haiku-5-5"


def list_bedrock_models(session, provider="Anthropic"):
    """The text models of this provider that this account's region offers (needs bedrock:ListFoundationModels)."""
    try:
        summaries = session.client("bedrock").list_foundation_models(byProvider=provider, byOutputModality="TEXT")["modelSummaries"]
    except (ClientError, BotoCoreError) as e:
        print(f"Could not list the models: {e}")
        return []
    return sorted(model["modelId"] for model in summaries)


def test_bedrock(session, model_id):
    """Asks the model one short question through the Converse API (needs bedrock:InvokeModel and the model enabled for the account)."""
    try:
        answer = session.client("bedrock-runtime").converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": "Reply with the single word: pong"}]}],
            inferenceConfig={"maxTokens": 20},
        )
        print(f"OK [{session.region_name}] {model_id}: {answer['output']['message']['content'][0]['text'].strip()}")
        print(f"tokens: {answer['usage']['inputTokens']} in, {answer['usage']['outputTokens']} out")
    except ClientError as e:
        error = e.response["Error"]
        print(f"ERROR {error['Code']}: {error['Message']}")
    except BotoCoreError as e:
        print(f"Connection ERROR: {e}")


models = list_bedrock_models(aws_session)
print(f"{len(models)} Anthropic models in {aws_session.region_name}:")
for model in models:
    print(f"  {model}")
print()
test_bedrock(aws_session, BEDROCK_MODEL)


16 Anthropic models in us-east-1:
  anthropic.claude-fable-5
  anthropic.claude-fable-5-1
  anthropic.claude-fable-5-2
  anthropic.claude-haiku-4-5-20251001-v1:0
  anthropic.claude-haiku-5-5
  anthropic.claude-opus-4-1-20250805-v1:0
  anthropic.claude-opus-4-5-20251101-v1:0
  anthropic.claude-opus-4-6-v1
  anthropic.claude-opus-4-7
  anthropic.claude-opus-4-8
  anthropic.claude-opus-5
  anthropic.claude-opus-5-5
  anthropic.claude-sonnet-4-5-20250929-v1:0
  anthropic.claude-sonnet-4-6
  anthropic.claude-sonnet-5
  anthropic.claude-sonnet-5-5

ERROR AccessDeniedException: anthropic.claude-haiku-5-5 is not available for this account. You can explore other available models on Amazon Bedrock. For additional access options, contact AWS Sales at https://aws.amazon.com/contact-us/sales-support/


In [3]:
try:
    from dotenv import set_key
except ImportError:
    def set_key(path, key, value):
        """Sets KEY="value" in a .env file: replaces the line of that key, or adds one. Same call as dotenv's set_key."""
        path = Path(path)
        lines = path.read_text(encoding="utf-8").splitlines() if path.exists() else []
        new = f'{key}="{value}"'
        for i, line in enumerate(lines):
            if line.split("=", 1)[0].strip() == key:
                lines[i] = new
                break
        else:
            lines.append(new)
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")

LOCAL_ENV = Path("undercoverlab/agentcore/.env.local")
LOCAL_ENV.touch()

secrets = {
    "AGENTCORE_CREDENTIAL_OPENROUTER": os.environ["OPENROUTER_API_KEY"],
    "AGENTCORE_CREDENTIAL_DATABASEURL": os.environ["DATABASE_URL"].replace(":5432/", ":6543/"),
}

for name, value in secrets.items():
    set_key(LOCAL_ENV, name, value)

print(f"Wrote {', '.join(secrets)} to {LOCAL_ENV}")


Wrote AGENTCORE_CREDENTIAL_OPENROUTER, AGENTCORE_CREDENTIAL_DATABASEURL to ..\undercoverlab\agentcore\.env.local


In [4]:
import json
from urllib.request import urlopen
import httpx

In [6]:


print(json.load(urlopen("http://127.0.0.1:8080/ping")))


{'status': 'Healthy', 'time_of_last_update': 1794484505}


In [7]:
question = {"action": "chat", "prompt": "What is 1,180 minus 240?"}

reply = httpx.post("http://127.0.0.1:8080/invocations", json=question, timeout=120)
print(reply.json())

{'answer': '1,180 − 240 = **940**.', 'session': None}


In [8]:
print(f"echo '{json.dumps(question)}' | agentcore invoke --runtime agent --payload -")

echo '{"action": "chat", "prompt": "What is 1,180 minus 240?"}' | agentcore invoke --runtime agent --payload -


In [10]:
agentcore = aws_session.client("bedrock-agentcore-control")


In [11]:
def find_runtime():
    """The lab's runtime, found by its name."""
    runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]
    return next(r for r in runtimes if r["agentRuntimeName"].startswith("undercoverlab"))


runtime = find_runtime()
runtime_arn = runtime["agentRuntimeArn"]
details = agentcore.get_agent_runtime(agentRuntimeId=runtime["agentRuntimeId"])
print(f"{runtime_arn} on platform version {details['platformVersion']}")

arn:aws:bedrock-agentcore:us-east-1:123456789012:runtime/undercoverlab_default_agent-R74ef0BBz1 on platform version V1


In [12]:
def new_session():
    """A new session id, long enough for Runtime."""
    return f"undercoverlab-{uuid.uuid4()}"


def call(payload, session_id):
    """Call the agent in a session, and print its reply and how long it took."""
    started = time.time()
    response = agentcore_runtime.invoke_agent_runtime(agentRuntimeArn=runtime_arn, runtimeSessionId=session_id, payload=json.dumps(payload))
    reply = json.loads(response["response"].read())
    print(f"{time.time() - started:.1f} seconds: {reply}")
    return reply

In [20]:
import uuid
import time
hello = {"action": "hello"}
session_id = new_session()

reply = call(hello, session_id)
reply = call(hello, session_id)

7.7 seconds: {'session': 'undercoverlab-5b65e036-0e48-4a8b-845d-bd17589eaaaf'}
0.2 seconds: {'session': 'undercoverlab-5b65e036-0e48-4a8b-845d-bd17589eaaaf'}


In [21]:
thread_id = str(uuid.uuid4())
first_session = new_session()

reply = call({"action": "start", "thread_id": thread_id}, first_session)

13.4 seconds: {'proposal': 'The delay qualifies, and the four receipts total $1,180, below the $1,500 limit. Pay $1,180, provided both taxi receipts are valid and the expenses are reasonable.', 'decision': None, 'paused': True, 'session': 'undercoverlab-ee5f3ab0-36ca-4bde-94d8-18946b26955e'}


In [22]:
response = agentcore_runtime.stop_runtime_session(agentRuntimeArn=runtime_arn, runtimeSessionId=first_session)
print(f"Stopped session {first_session}")

Stopped session undercoverlab-ee5f3ab0-36ca-4bde-94d8-18946b26955e


In [24]:
import psycopg

with psycopg.connect(os.environ["DATABASE_URL"]) as connection:
    rows = connection.execute("SELECT checkpoint_id FROM lab.checkpoints WHERE thread_id = %s", (thread_id,)).fetchall()

print(f"{len(rows)} checkpoints saved on thread {thread_id}")

3 checkpoints saved on thread 6440a205-d410-4168-a526-0def1d96d14b


In [25]:
second_session = new_session()
note = {"action": "resume", "thread_id": thread_id, "note": "The taxi receipt is in twice."}

reply = call(note, second_session)

13.6 seconds: {'proposal': 'The delay qualifies, and the four receipts total $1,180, below the $1,500 limit. Pay $1,180, provided both taxi receipts are valid and the expenses are reasonable.', 'decision': 'The delay qualifies, but one taxi receipt is duplicated, so the eligible expenses total $940. Pay $940, provided the remaining expenses are reasonable and valid.', 'paused': False, 'session': 'undercoverlab-4aa1d9a9-401d-4f58-8782-0dc3534e0bfd'}


In [26]:
LOG_GROUPS = ["/aws/bedrock-agentcore/runtimes/hellolab", "/aws/bedrock-agentcore/runtimes/undercoverlab"]

for prefix in LOG_GROUPS:
    for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=prefix):
        for group in page["logGroups"]:
            logs.delete_log_group(logGroupName=group["logGroupName"])
            print(f"Deleted log group {group['logGroupName']}")

Deleted log group /aws/bedrock-agentcore/runtimes/hellolab_default_agent-wddXdF4fXJ-DEFAULT
Deleted log group /aws/bedrock-agentcore/runtimes/undercoverlab_default_agent-R74ef0BBz1-DEFAULT


In [27]:
def show_resources():
    """Print what this lab created that still exists."""
    stacks = [s["StackName"] for page in cloudformation.get_paginator("describe_stacks").paginate() for s in page["Stacks"]]
    providers = agentcore.list_api_key_credential_providers()["credentialProviders"]
    projects = ("hellolab", "undercoverlab")
    found = {
        "Runtimes": [r["agentRuntimeName"] for r in agentcore.list_agent_runtimes()["agentRuntimes"] if r["agentRuntimeName"].startswith(projects)],
        "Credentials": [p["name"] for p in providers if p["name"].startswith(projects)],
        "Stacks": [name for name in stacks if name.startswith(("AgentCore-hellolab", "AgentCore-undercoverlab", "CDKToolkit"))],
    }
    for label, names in found.items():
        print(f"{label + ':':<14}{', '.join(names) or 'none'}")


show_resources()

Runtimes:     none
Credentials:  none
Stacks:       CDKToolkit
